In [3]:
# LINKING_V5 — L1: load raw ROI paths + imaging sheet

from pathlib import Path
import pandas as pd

# ─── paths ───
ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v3"
RAW  = BASE / "raw"
WORK = BASE / "working"

for p in [ROOT, BASE, RAW, WORK]:
    if not p.exists():
        raise FileNotFoundError(f"Expected directory at {p}, but it does not exist.")

ROI_PATHS_FILE     = RAW / "2025-11-13-092338-korra_aang_roi_root_tiffs_good-3.xlsx"
IMAGING_SHEET_FILE = RAW / "2025-11-21-220012-imaging_sheet.xlsx"

print("ROOT:", ROOT)
print("ROI paths file:     ", ROI_PATHS_FILE)
print("Imaging sheet file: ", IMAGING_SHEET_FILE)

if not ROI_PATHS_FILE.exists():
    raise FileNotFoundError(f"ROI paths file not found at {ROI_PATHS_FILE}")
if not IMAGING_SHEET_FILE.exists():
    raise FileNotFoundError(f"Imaging sheet file not found at {IMAGING_SHEET_FILE}")

# ─── load raw tables ───
df_roi_paths = pd.read_excel(ROI_PATHS_FILE)
df_imaging   = pd.read_excel(IMAGING_SHEET_FILE)

print("\nL1 — df_roi_paths shape:", df_roi_paths.shape)
print("L1 — df_imaging shape: ", df_imaging.shape)

print("\nL1 — df_roi_paths columns:")
print(list(df_roi_paths.columns))

print("\nL1 — df_imaging columns:")
print(list(df_imaging.columns))

ROOT: /Users/davekokel/Projects/carp_v2
ROI paths file:      /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/2025-11-13-092338-korra_aang_roi_root_tiffs_good-3.xlsx
Imaging sheet file:  /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/2025-11-21-220012-imaging_sheet.xlsx

L1 — df_roi_paths shape: (976, 7)
L1 — df_imaging shape:  (347, 28)

L1 — df_roi_paths columns:
['date_experiment', 'fish', 'roi_rel', 'roi_name', 'roi_tiffs', 'roi_dir', 'dataset']

L1 — df_imaging columns:
['date_mount', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Time mounted', 'Mounting Orientation', 'Date screened/Initial feedback', 'Date imaged', 'Time placed in scope', 'Start of imaging time', 'End of imaging time', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'Dataset size (GB) - raw data

In [4]:
# LINKING_V5 — L2: parse roi_dir into date/slug/fish/roi_folder, infer fish_id and roi_anatomy

import re
import pandas as pd

df_roi = df_roi_paths.copy()

# ─────────────────────────────────────────────────────────────
# L2a — basic path normalization
# ─────────────────────────────────────────────────────────────

def norm_path(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    # normalize slashes
    s = s.replace("\\", "/")
    # collapse multiple slashes
    s = re.sub(r"/+", "/", s)
    return s.rstrip("/")

df_roi["roi_dir"] = df_roi["roi_dir"].apply(norm_path)

# drop any rows with missing roi_dir up front (they are not real ROIs)
n_before = len(df_roi)
df_roi = df_roi[df_roi["roi_dir"].notna()].copy()
n_after = len(df_roi)
if n_after != n_before:
    print(f"L2a: dropped {n_before - n_after} rows with null roi_dir (non-ROI artifacts).")

# ─────────────────────────────────────────────────────────────
# L2b — extract experiment_folder, date, fish_folder, roi_folder
# ─────────────────────────────────────────────────────────────

def parse_roi_hierarchy(path: str):
    """
    Given a roi_dir like:
      /clusterfs/vast/abcabc/Korra_Foundation/20250428_mem_histone/fish1_72hpf/roi1_tail
    return:
      experiment_folder = '20250428_mem_histone'
      roi_path_date_yyyymmdd = '20250428'
      fish_folder = 'fish1_72hpf'
      roi_folder  = 'roi1_tail'
    """
    if not path:
        return None, None, None, None

    parts = path.split("/")
    try:
        # find the Foundation segment and take the next as experiment_folder
        for i, p in enumerate(parts):
            if p in ("Korra_Foundation", "Aang_Foundation"):
                experiment_folder = parts[i + 1] if i + 1 < len(parts) else None
                # date prefix from experiment_folder if present
                m = re.match(r"^(\d{8})_", experiment_folder) if experiment_folder else None
                roi_date = m.group(1) if m else None
                # fish_folder = next segment after experiment_folder
                fish_folder = parts[i + 2] if i + 2 < len(parts) else None
                # roi_folder = last segment
                roi_folder = parts[-1] if parts else None
                return experiment_folder, roi_date, fish_folder, roi_folder
        # if we didn't find Foundation, fall back to last 3 segments
        experiment_folder = parts[-3] if len(parts) >= 3 else None
        m = re.match(r"^(\d{8})_", experiment_folder) if experiment_folder else None
        roi_date = m.group(1) if m else None
        fish_folder = parts[-2] if len(parts) >= 2 else None
        roi_folder  = parts[-1] if parts else None
        return experiment_folder, roi_date, fish_folder, roi_folder
    except Exception:
        return None, None, None, None

parsed = df_roi["roi_dir"].apply(parse_roi_hierarchy)
df_roi["experiment_folder"]        = parsed.apply(lambda x: x[0])
df_roi["roi_path_date_yyyymmdd"]  = parsed.apply(lambda x: x[1])
df_roi["fish_folder"]             = parsed.apply(lambda x: x[2])
df_roi["roi_folder"]              = parsed.apply(lambda x: x[3])

print("L2b — sample hierarchy fields from ROI paths:")
print(
    df_roi[["roi_dir", "roi_path_date_yyyymmdd", "experiment_folder", "fish_folder", "roi_folder"]]
    .head(20)
)

# ─────────────────────────────────────────────────────────────
# L2c — infer fish_id, fish_number, fish_age_hpf, fish_nickname
# ─────────────────────────────────────────────────────────────

def parse_fish_folder(fish_folder: str):
    """
    Examples:
      'fish1_72hpf'              → fish_id='fish1',  fish_number=1,  age='72', nickname=None
      'fish3_mem-halo_24hpf'     → fish_id='fish3',  fish_number=3,  age='24', nickname='mem-halo'
      'fish10'                   → fish_id='fish10', fish_number=10, age=None, nickname=None
      'fish7_roi2' (we still treat as fish7, ignore extra tokens for age/nickname)
    """
    if not fish_folder:
        return None, None, None, None

    s = str(fish_folder)

    # 1) find fish number anywhere in the string
    m = re.search(r"fish(\d+)", s, flags=re.IGNORECASE)
    if not m:
        return None, None, None, None

    num = int(m.group(1))
    fish_id = f"fish{num}"

    # 2) tokenize on '_' and look for age token (like '24hpf', '72hpf')
    tokens = s.split("_")
    age_hpf = None
    nickname = None

    # find age token (ends with 'hpf')
    age_idx = None
    for i, tok in enumerate(tokens):
        tok_clean = tok.lower()
        if tok_clean.endswith("hpf"):
            # strip 'hpf'
            digits = re.match(r"(\d+)", tok_clean)
            if digits:
                age_hpf = int(digits.group(1))
                age_idx = i
                break

    # nickname = everything between the 'fishN...' token and the age token
    # e.g. fish3_mem-halo_24hpf → nickname='mem-halo'
    if age_idx is not None:
        # find index of token that contains 'fishN'
        fish_tok_idx = None
        for i, tok in enumerate(tokens):
            if re.search(r"fish\d+", tok, flags=re.IGNORECASE):
                fish_tok_idx = i
                break
        if fish_tok_idx is not None and age_idx - fish_tok_idx > 1:
            nickname_tokens = tokens[fish_tok_idx + 1 : age_idx]
            nickname = "_".join(nickname_tokens) if nickname_tokens else None

    return fish_id, float(num), age_hpf, nickname

fish_parsed = df_roi["fish_folder"].apply(parse_fish_folder)
df_roi["fish_id"]       = fish_parsed.apply(lambda x: x[0])
df_roi["fish_number"]   = fish_parsed.apply(lambda x: x[1])
df_roi["fish_age_hpf"]  = fish_parsed.apply(lambda x: x[2])
df_roi["fish_nickname"] = fish_parsed.apply(lambda x: x[3])

print("\nL2c — QC: fish_id vs raw 'fish' column")
print("  total ROI rows:", len(df_roi))
print("  unique fish_id:", df_roi["fish_id"].nunique())
print("  unique fish_folder:", df_roi["fish_folder"].nunique())
print("  unique raw 'fish' values:", df_roi["fish"].nunique())

# compare fish_id to raw 'fish' (normalized)
def norm_raw_fish(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    # for rows where 'fish' is like 'fish1' or 'fish1_24hpf_roi1', extract fishN
    m = re.search(r"fish(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"fish{int(m.group(1))}"
    return s

df_roi["fish_raw_norm"] = df_roi["fish"].apply(norm_raw_fish)
mask_nonnull_raw = df_roi["fish_raw_norm"].notna()
matches = (df_roi.loc[mask_nonnull_raw, "fish_id"] == df_roi.loc[mask_nonnull_raw, "fish_raw_norm"])
print("  rows with non-null raw fish:", mask_nonnull_raw.sum())
print("  fish_id matches raw:", matches.sum())
print("  fish_id mismatches:", (~matches).sum())

if (~matches).sum() > 0:
    print("\nL2c — sample mismatches (fish_folder, fish_raw_norm, fish_id):")
    print(
        df_roi.loc[
            mask_nonnull_raw & ~matches,
            ["roi_dir", "fish_folder", "fish", "fish_raw_norm", "fish_id"]
        ].head(20)
    )

print("\nL2c — distinct fish_folder -> fish_id (first 40):")
print(
    df_roi[["experiment_folder", "fish_folder", "fish_number", "fish_id", "fish_age_hpf", "fish_nickname"]]
    .drop_duplicates()
    .head(40)
)

# ─────────────────────────────────────────────────────────────
# L2d — infer roi_anatomy from roi_folder (token-based)
# ─────────────────────────────────────────────────────────────

ANATOMY_TOKENS = {
    "brain": "brain",
    "hindbrain": "hindbrain",
    "ear": "ear",
    "eye": "eye",
    "spine": "spine",
    "spinal": "spinal_cord",
    "cord": "spinal_cord",
    "muscle": "muscle",
    "tail": "tail",
    "tailbud": "tailbud",
    "skin": "skin",
    "neuromast": "neuromast",
    "notochord": "notochord",
    "blood": "blood_vessel",
    "vessel": "blood_vessel",
}

def parse_roi_anatomy(roi_folder: str):
    if not roi_folder:
        return [], None
    s = str(roi_folder).lower()
    # split on underscores and other separators
    raw_tokens = re.split(r"[_\-\s]+", s)
    tokens = [t for t in raw_tokens if t]
    hits = []
    for t in tokens:
        for key, canon in ANATOMY_TOKENS.items():
            if key in t:
                hits.append(canon)
    # dedupe preserving order
    seen = set()
    uniq = []
    for h in hits:
        if h not in seen:
            seen.add(h)
            uniq.append(h)
    if not uniq:
        return [], None
    return uniq, "|".join(uniq)

roi_ana = df_roi["roi_folder"].apply(parse_roi_anatomy)
df_roi["roi_anatomy_tokens"] = roi_ana.apply(lambda x: x[0])
df_roi["roi_anatomy"]        = roi_ana.apply(lambda x: x[1])

print("\nL2d — sample roi_folder -> roi_anatomy_tokens / roi_anatomy:")
print(
    df_roi[["roi_dir", "roi_folder", "roi_anatomy_tokens", "roi_anatomy"]]
    .head(40)
)

print("\nL2d — distinct roi_folder -> roi_anatomy (first 40):")
print(
    df_roi[["experiment_folder", "roi_folder", "roi_anatomy"]]
    .drop_duplicates()
    .head(40)
)

L2b — sample hierarchy fields from ROI paths:
                                              roi_dir roi_path_date_yyyymmdd  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250428   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250428   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
8   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
9   /clusterfs/vast/abcabc/Korra_Foundation/202504...               20250429   
10  /clusterfs/vast/abcabc/Korra_Foundation/202505...               202505

In [5]:
# LINKING_V5 — L2e: patch malformed fish_folder cases

df_roi = df_roi.copy()

# 1. patterns that really mean "fish1", "fish2", etc.
#    based on consistent datasets you showed earlier

PATCH_MAP = {
    # peroxi / mem-histone misplacements:
    # these slugs usually have fish numbers encoded in roi_folder
    r"^20251028_peroxi$": None,
    r"^20251029_peroxi$": None,
    r"^20251029_mem-histone$": None,

    # “fish_48hpf_roi1” → no number, but the ROI folder *does* include roiN
    r"^fish_48hpf_roi\d+$": None,

    # “er_roi1”, “mito_roi2”, etc — these also have no fish number
    r"^[a-zA-Z]+_roi\d+$": None,

    # “roi1_newscripttest”
    r"^roi\d+.*$": None,
}

import re

def patch_fish_folder_badcases(row):
    fish_folder = row["fish_folder"]
    roi_folder  = row["roi_folder"]
    exp_folder  = row["experiment_folder"]

    if fish_folder is None:
        return None

    # cases with no fish number at all
    for pat in PATCH_MAP:
        if re.match(pat, str(fish_folder)):
            # Try to infer fishN from roi_folder
            m = re.search(r"roi(\d+)", roi_folder or "")
            if m:
                n = int(m.group(1))
                return f"fish{n}"
            # Try infer from experiment by counting ROIs later
            return None

    return fish_folder

df_roi["fish_folder_patched"] = df_roi.apply(patch_fish_folder_badcases, axis=1)

# Now recompute fish_id from patched folder:
patched = df_roi["fish_folder_patched"].apply(parse_fish_folder)
df_roi["fish_id"]       = patched.apply(lambda x: x[0])
df_roi["fish_number"]   = patched.apply(lambda x: x[1])
df_roi["fish_age_hpf"]  = patched.apply(lambda x: x[2])
df_roi["fish_nickname"] = patched.apply(lambda x: x[3])

print("L2e — AFTER PATCH: fish_id missing =", df_roi["fish_id"].isna().sum())
print("L2e — sample rows with patched fish_folder:")
print(
    df_roi.loc[df_roi["fish_id"].isna() | df_roi["fish_folder"] != df_roi["fish_folder_patched"],
               ["roi_dir", "fish_folder", "fish_folder_patched", "roi_folder", "fish_id"]]
    .head(20)
)

L2e — AFTER PATCH: fish_id missing = 0
L2e — sample rows with patched fish_folder:
                                              roi_dir           fish_folder  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...           fish1_72hpf   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...           fish1_72hpf   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...           fish1_24hpf   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...           fish1_24hpf   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...           fish2_48hpf   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...  fish3_mem-halo_24hpf   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...  fish3_mem-halo_24hpf   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...  fish4_mem-halo_48hpf   
8   /clusterfs/vast/abcabc/Korra_Foundation/202504...  fish4_mem-halo_48hpf   
9   /clusterfs/vast/abcabc/Korra_Foundation/202504...  fish4_mem-halo_48hpf   
10  /clusterfs/vast/abcabc/Korra_Foundation/2025

In [6]:
# LINKING_V5 — L3: normalize imaging sheet (slug + date_mount_yyyymmdd)

import re
import pandas as pd

# We assume df_imaging already loaded in L1, df_roi from L2/L2e.

df_imaging = df_imaging.copy()

print("L3 — starting from df_imaging shape:", df_imaging.shape)
print("L3 — df_imaging columns:", list(df_imaging.columns))

# 1) Derive sheet_slug from Data location
def slug_from_data_location(s: str) -> str:
    """
    Imaging sheet paths look like:
      X:\\abcabc\\Korra_Foundation\\20250513_skittles(fish1-3)
    We want the '20250513_skittles' part (everything after Foundation up to first backslash or paren).
    """
    if pd.isna(s):
        return None
    s = str(s)
    # Capture after Foundation\
    m = re.search(r"\\(Korra_Foundation|Aang_Foundation)\\([^\\]+)", s)
    if not m:
        return None
    slug = m.group(2)
    # strip trailing "(fish1-3)" or similar and whitespace
    slug = re.split(r"[()\[]", slug, maxsplit=1)[0]
    return slug.strip()

df_imaging["sheet_slug"] = df_imaging["Data location"].apply(slug_from_data_location)

# 2) Derive date_mount_yyyymmdd from the date_mount column

# Try to find a mount-date-like column
date_mount_col = None
for cand in ["date_mount", "Date mount", "Date mounted", "Date mounted?"]:
    if cand in df_imaging.columns:
        date_mount_col = cand
        break

if date_mount_col is None:
    raise KeyError("L3: Could not find a date_mount-like column in df_imaging.")

df_imaging["date_mount_yyyymmdd"] = pd.to_datetime(
    df_imaging[date_mount_col],
    errors="coerce"
).dt.strftime("%Y%m%d")

# 3) Normalize slug for joining with ROI

def norm_slug(s):
    if pd.isna(s):
        return None
    return str(s).strip().lower()

df_imaging["sheet_slug_norm"] = df_imaging["sheet_slug"].apply(norm_slug)

print("\nL3 — imaging slug/date preview:")
print(
    df_imaging[
        [
            "sheet_slug",
            "sheet_slug_norm",
            date_mount_col,
            "date_mount_yyyymmdd",
            "Data location",
        ]
    ].head(20)
)

# 4) QC — how many (slug_norm, date_mount_yyyymmdd) combos?

combo_counts = (
    df_imaging[["sheet_slug_norm", "date_mount_yyyymmdd"]]
    .value_counts(dropna=False)
    .reset_index(name="n_rows")
)

print("\nL3 — unique (sheet_slug_norm, date_mount_yyyymmdd) combos:", len(combo_counts))
print("L3 — sample combos:")
print(combo_counts.head(30))

L3 — starting from df_imaging shape: (347, 28)
L3 — df_imaging columns: ['date_mount', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Time mounted', 'Mounting Orientation', 'Date screened/Initial feedback', 'Date imaged', 'Time placed in scope', 'Start of imaging time', 'End of imaging time', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'Dataset size (GB) - raw data only', 'Camera Filters', 'JSON excite map for ZF male', 'JSON excite map for ZF female', 'JSON excite map for plasmid', 'JSON excite map for mRNA', 'comments', 'Data evaluation comments']

L3 — imaging slug/date preview:
                  sheet_slug           sheet_slug_norm date_mount  \
0                       None                      None 2025-11-21   
1       20251121_mem-histone      20251121_mem-histone 2025-11-21   
2       20251117_me

In [8]:
# LINKING_V5 — L4: join ROI paths to imaging sheet on (slug_norm, date_mount_yyyymmdd)

import pandas as pd
import re
from pathlib import Path

# ─────────────────────────────────────────────
# 0) Ensure WORKING is defined (in case L1 wasn't run in this kernel)
# ─────────────────────────────────────────────
if "WORKING" not in globals():
    ROOT = Path("/Users/davekokel/Projects/carp_v2")
    BASE = ROOT / "seed_kits" / "legacy_wrangling_v3"
    RAW = BASE / "raw"
    WORKING = BASE / "working"

print("L4 — starting join")
print("  df_roi shape:     ", df_roi.shape)
print("  df_imaging shape: ", df_imaging.shape)

# ─────────────────────────────────────────────
# 1) Normalize ROI-side slug for join
# ─────────────────────────────────────────────

def norm_slug(s):
    if pd.isna(s):
        return None
    return str(s).strip().lower()

df_roi = df_roi.copy()

# dataset_slug on ROI side is just experiment_folder
df_roi["dataset_slug"]      = df_roi["experiment_folder"]
df_roi["dataset_slug_norm"] = df_roi["dataset_slug"].apply(norm_slug)

print("\nL4 — ROI slug/date preview:")
print(
    df_roi[
        ["roi_dir", "experiment_folder", "dataset_slug", "dataset_slug_norm", "roi_path_date_yyyymmdd"]
    ].head(20)
)

# ─────────────────────────────────────────────
# 2) Build a slim imaging table for the join
# ─────────────────────────────────────────────

im_cols_keep = [
    "sheet_slug_norm",
    "date_mount_yyyymmdd",
    "date_mount",
    "Date imaged",
    "mount_id",
    "ZF female genotype",
    "ZF male genotype",
    "additional plasmids injected",
    "additional mRNAs injected",
    "additonal proteins injected",
    "additonal dye and chemicals",
    "Date born",
    "Imaged Locations",
    "Unique Targets with blanks",
    "Unique Targets",
    "Data location",
]

im_cols_keep = [c for c in im_cols_keep if c in df_imaging.columns]
df_imaging_slim = df_imaging[im_cols_keep].copy()

print("\nL4 — imaging_slim columns:", df_imaging_slim.columns.tolist())

# ─────────────────────────────────────────────
# 3) Join ROI ↔ imaging on (slug_norm, date_mount_yyyymmdd)
#     ROI:    (dataset_slug_norm, roi_path_date_yyyymmdd)
#     SHEET:  (sheet_slug_norm,   date_mount_yyyymmdd)
# ─────────────────────────────────────────────

df_linked = df_roi.merge(
    df_imaging_slim,
    left_on=["dataset_slug_norm", "roi_path_date_yyyymmdd"],
    right_on=["sheet_slug_norm", "date_mount_yyyymmdd"],
    how="left",
    indicator=True,
)

# link_source: 'sheet' if joined, 'unmatched' otherwise
df_linked["link_source"] = df_linked["_merge"].map(
    {"both": "sheet", "left_only": "unmatched", "right_only": "imaging_only"}
)
df_linked = df_linked.drop(columns=["_merge"])

print("\nL4 — df_linked shape:", df_linked.shape)
print("L4 — unique roi_dir:", df_linked["roi_dir"].nunique())
print("L4 — link_source breakdown:")
print(df_linked["link_source"].value_counts())

# ─────────────────────────────────────────────
# 4) QC: show a sample of unmatched ROIs and optionally write them
# ─────────────────────────────────────────────

unmatched = df_linked[df_linked["link_source"] == "unmatched"]
print("\nL4 — unmatched ROI rows:", len(unmatched))

if len(unmatched):
    print("L4 — sample unmatched ROIs:")
    print(
        unmatched[
            [
                "roi_dir",
                "experiment_folder",
                "dataset_slug_norm",
                "roi_path_date_yyyymmdd",
            ]
        ].head(30)
    )

    try:
        WORKING.mkdir(parents=True, exist_ok=True)
        out_unmatched = WORKING / "linking_v5_unmatched_slug_date.csv"
        unmatched[
            [
                "roi_dir",
                "experiment_folder",
                "dataset_slug_norm",
                "roi_path_date_yyyymmdd",
            ]
        ].to_csv(out_unmatched, index=False)
        print("\nL4 — wrote unmatched ROI list to:", out_unmatched)
    except Exception as e:
        print("\nL4 — WARNING: could not write unmatched ROI list:", e)

print("\nL4 — done.")

L4 — starting join
  df_roi shape:      (976, 21)
  df_imaging shape:  (347, 31)

L4 — ROI slug/date preview:
                                              roi_dir     experiment_folder  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250428_mem_histone   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250428_mem_histone   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
5   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
6   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
7   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
8   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
9   /clusterfs/vast/abcabc/Korra_Foundation/202504...  20250429_mem_cytosol   
10  /clusterfs/vast/a

In [10]:
# LINKING_V5 — L5: date-only fallback linking for unmatched ROIs (rule C)

import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────
# 0) Ensure we have WORKING and df_imaging ready
# ─────────────────────────────────────────────
if "WORKING" not in globals():
    ROOT = Path("/Users/davekokel/Projects/carp_v2")
    BASE = ROOT / "seed_kits" / "legacy_wrangling_v3"
    WORKING = BASE / "working"
WORKING.mkdir(parents=True, exist_ok=True)

print("L5 — starting date-only fallback")
print("  df_linked shape:", df_linked.shape)

# Ensure roi_path_date_yyyymmdd exists
if "roi_path_date_yyyymmdd" not in df_linked.columns:
    raise KeyError("L5: df_linked is missing 'roi_path_date_yyyymmdd' from L2b.")

# Ensure df_imaging has date_mount_yyyymmdd
if "date_mount_yyyymmdd" not in df_imaging.columns:
    if "date_mount" not in df_imaging.columns:
        raise KeyError("L5: df_imaging has neither 'date_mount_yyyymmdd' nor 'date_mount'.")
    df_imaging["date_mount_yyyymmdd"] = pd.to_datetime(
        df_imaging["date_mount"], errors="coerce"
    ).dt.strftime("%Y%m%d")

# ─────────────────────────────────────────────
# 1) Identify unmatched ROI rows
# ─────────────────────────────────────────────
mask_unmatched = df_linked["link_source"] == "unmatched"
df_holes = df_linked[mask_unmatched].copy()

print("L5 — unmatched ROI rows (before fallback):", len(df_holes))

# restrict to holes that have a parsable date from roi_path
df_holes = df_holes[~df_holes["roi_path_date_yyyymmdd"].isna()].copy()
print("L5 — holes with roi_path_date_yyyymmdd:", len(df_holes))

# ─────────────────────────────────────────────
# 2) Count imaging rows per date_mount_yyyymmdd
#    and define "safe" dates = exactly ONE imaging row
# ─────────────────────────────────────────────
imaging_counts = (
    df_imaging["date_mount_yyyymmdd"]
    .value_counts(dropna=False)
    .rename_axis("date_mount_yyyymmdd")
    .reset_index(name="n_rows")
)

safe_dates = set(
    imaging_counts.loc[imaging_counts["n_rows"] == 1, "date_mount_yyyymmdd"].dropna()
)

print("L5 — number of dates with exactly one imaging row:", len(safe_dates))

# hole dates
hole_dates = set(df_holes["roi_path_date_yyyymmdd"].dropna().astype(str).unique())
safe_hole_dates = hole_dates & safe_dates
print("L5 — number of hole dates that are 'safe' (one imaging row):", len(safe_hole_dates))

# ─────────────────────────────────────────────
# 3) Build a date→imaging row lookup for safe dates
# ─────────────────────────────────────────────
im_cols_keep = [
    "sheet_slug_norm",
    "date_mount_yyyymmdd",
    "date_mount",
    "Date imaged",
    "mount_id",
    "ZF female genotype",
    "ZF male genotype",
    "additional plasmids injected",
    "additional mRNAs injected",
    "additonal proteins injected",
    "additonal dye and chemicals",
    "Date born",
    "Imaged Locations",
    "Unique Targets with blanks",
    "Unique Targets",
    "Data location",
]
im_cols_keep = [c for c in im_cols_keep if c in df_imaging.columns]

safe_imaging = (
    df_imaging[df_imaging["date_mount_yyyymmdd"].isin(safe_hole_dates)][im_cols_keep]
    .drop_duplicates(subset=["date_mount_yyyymmdd"])
    .set_index("date_mount_yyyymmdd")
)

print("L5 — safe_imaging rows:", safe_imaging.shape[0])

# ─────────────────────────────────────────────
# 4) Apply date-only fallback to df_linked
# ─────────────────────────────────────────────
df_linked = df_linked.copy()

mask_candidate = (
    (df_linked["link_source"] == "unmatched")
    & df_linked["roi_path_date_yyyymmdd"].notna()
    & df_linked["roi_path_date_yyyymmdd"].astype(str).isin(safe_hole_dates)
)

print("L5 — candidate rows for date-only fallback:", mask_candidate.sum())

def _apply_date_fallback(row):
    if not mask_candidate.loc[row.name]:
        return row
    date_key = str(row["roi_path_date_yyyymmdd"])
    if date_key not in safe_imaging.index:
        return row
    im_row = safe_imaging.loc[date_key]
    for col in im_cols_keep:
        if col == "date_mount_yyyymmdd":
            continue
        if col in im_row.index:
            row[col] = im_row[col]
    row["link_source"] = "date_match"
    return row

df_linked = df_linked.apply(_apply_date_fallback, axis=1)

# ─────────────────────────────────────────────
# 5) QC + write remaining holes needing manual review
# ─────────────────────────────────────────────
print("\nL5 — link_source breakdown after date-only fallback:")
print(df_linked["link_source"].value_counts())

remaining_holes = df_linked[df_linked["link_source"] == "unmatched"].copy()
print("L5 — remaining unmatched ROI rows after fallback:", len(remaining_holes))

out_remaining = WORKING / "linking_v5_hole_rows_needing_manual_review.csv"
remaining_holes[
    [
        "roi_dir",
        "experiment_folder",
        "dataset_slug_norm",
        "roi_path_date_yyyymmdd",
    ]
].to_csv(out_remaining, index=False)
print("L5 — wrote remaining unmatched ROI rows to:", out_remaining)

print("\nL5 — done.")

L5 — starting date-only fallback
  df_linked shape: (1082, 38)
L5 — unmatched ROI rows (before fallback): 155
L5 — holes with roi_path_date_yyyymmdd: 144
L5 — number of dates with exactly one imaging row: 74
L5 — number of hole dates that are 'safe' (one imaging row): 6
L5 — safe_imaging rows: 6
L5 — candidate rows for date-only fallback: 43

L5 — link_source breakdown after date-only fallback:
link_source
sheet         927
unmatched     112
date_match     43
Name: count, dtype: int64
L5 — remaining unmatched ROI rows after fallback: 112
L5 — wrote remaining unmatched ROI rows to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/linking_v5_hole_rows_needing_manual_review.csv

L5 — done.


In [13]:
# LINKING_V5 — L6: infer mount/plate/slot/ROI IDs with safe fallbacks
# - plate_date from: date_mount → roi_path_date_yyyymmdd → experiment_folder prefix
# - rows with no date at all are logged and carried through with NA plate/mount IDs

import pandas as pd
import re

df_linked = df_linked.copy()

# ─────────────────────────────────────────────
# 1) plate_date with 3-level fallback
# ─────────────────────────────────────────────

# 1a) date_mount → YYYYMMDD (if present)
if "date_mount" in df_linked.columns:
    dt_mount_str = pd.to_datetime(df_linked["date_mount"], errors="coerce").dt.strftime("%Y%m%d")
else:
    dt_mount_str = pd.Series([None] * len(df_linked), index=df_linked.index)

# 1b) roi_path_date_yyyymmdd from ROI path
roi_date_str = df_linked["roi_path_date_yyyymmdd"].astype(str).where(
    df_linked["roi_path_date_yyyymmdd"].notna(), None
)

# 1c) experiment_folder prefix: ^(\d{8})_
def extract_exp_date(s):
    if pd.isna(s):
        return None
    m = re.match(r"^(\d{8})_", str(s))
    return m.group(1) if m else None

exp_date_str = df_linked["experiment_folder"].apply(extract_exp_date)

# combine: imaging date → ROI path date → experiment_folder date
plate_date = dt_mount_str.where(dt_mount_str.notna(), roi_date_str)
plate_date = plate_date.where(plate_date.notna(), exp_date_str)

df_linked["plate_date"] = plate_date

missing_plate_date_mask = df_linked["plate_date"].isna()
n_missing_plate_date = missing_plate_date_mask.sum()
print("L6 — rows with missing plate_date AFTER all fallbacks:", n_missing_plate_date)

if n_missing_plate_date:
    bad = df_linked[missing_plate_date_mask]
    out_bad = WORKING / "linking_v5_missing_plate_date_rows.csv"
    bad[[
        "roi_dir",
        "experiment_folder",
        "dataset_slug_norm",
        "roi_path_date_yyyymmdd",
        "date_mount",
        "Date imaged",
        "Data location",
        "link_source",
    ]].to_csv(out_bad, index=False)
    print("L6 — wrote rows with missing plate_date to:", out_bad)
    print("L6 — will carry these rows through with NA mount/plate/slot IDs (manual review).")

# Split into good/bad for inference
df_good = df_linked[~missing_plate_date_mask].copy()
df_bad  = df_linked[missing_plate_date_mask].copy()

# ─────────────────────────────────────────────
# 2) infer mount_id_inferred from fish_id per (plate_date, experiment_folder)
#    (only on df_good)
# ─────────────────────────────────────────────

if "fish_id" not in df_good.columns:
    raise KeyError("L6: df_linked missing 'fish_id'; make sure L2c/L2e ran correctly.")

def infer_mount_id_per_group(fish_ids: pd.Series) -> pd.Series:
    """
    For a given (plate_date, experiment_folder) group, assign mount IDs so that
    each mount has up to 6 unique fish_id entries.

    - Get sorted unique fish_id
    - Map each fish_id to index 0..N-1
    - mount_id_inferred = idx // 6 + 1
    """
    uniques = sorted(fish_ids.unique())
    idx_map = {f: i for i, f in enumerate(uniques)}
    idx = fish_ids.map(idx_map)
    return (idx // 6) + 1

df_good["mount_id_inferred"] = (
    df_good
    .sort_values(["plate_date", "experiment_folder", "fish_id"])
    .groupby(["plate_date", "experiment_folder"])["fish_id"]
    .transform(infer_mount_id_per_group)
)
df_good["mount_id_source"] = "inferred"

# ─────────────────────────────────────────────
# 3) plate_id_filled, slot_id_filled, roi_index_within_slot (df_good only)
# ─────────────────────────────────────────────

df_good["plate_key"] = (
    df_good["plate_date"].astype(str)
    + "::" + df_good["experiment_folder"].astype(str)
    + "::" + df_good["mount_id_inferred"].astype(int).astype(str)
)

df_good["plate_id_filled"] = pd.factorize(df_good["plate_key"])[0] + 1

df_good["slot_id_filled"] = (
    df_good
    .sort_values(["plate_id_filled", "fish_id"])
    .groupby("plate_id_filled")["fish_id"]
    .transform(lambda s: pd.factorize(s, sort=True)[0] + 1)
)

df_good["roi_index_within_slot"] = (
    df_good
    .sort_values(["plate_id_filled", "slot_id_filled", "roi_folder"])
    .groupby(["plate_id_filled", "slot_id_filled"])["roi_folder"]
    .cumcount() + 1
)

# ─────────────────────────────────────────────
# 4) QC on df_good: ≤ 6 fish per inferred plate
# ─────────────────────────────────────────────

fish_counts = (
    df_good
    .groupby(["plate_date", "experiment_folder", "mount_id_inferred", "plate_id_filled"])["fish_id"]
    .nunique()
    .reset_index(name="n_fish")
)

bad_plates = fish_counts[fish_counts["n_fish"] > 6]
print("L6 — inferred plates with >6 fish_id (should be 0):", len(bad_plates))
if len(bad_plates):
    out_bad_plates = WORKING / "linking_v5_bad_plates_n_fish_gt6_INFERRED.csv"
    bad_plates.to_csv(out_bad_plates, index=False)
    print("L6 — wrote bad inferred plates summary to:", out_bad_plates)
    raise ValueError("L6: found inferred plates with >6 fish; check inference logic or raw data.")

# ─────────────────────────────────────────────
# 5) Bruker ROI ID (df_good only)
# ─────────────────────────────────────────────

def make_bruker_roi_id(row):
    date_str = str(row["plate_date"])
    plate    = int(row["plate_id_filled"])
    slot     = int(row["slot_id_filled"])
    idx      = int(row["roi_index_within_slot"])
    return f"{date_str}-plate{plate}-slot{slot}-roi{idx}"

df_good["bruker_roi_id"] = df_good.apply(make_bruker_roi_id, axis=1)

# ─────────────────────────────────────────────
# 6) Prepare df_bad: carry forward, but leave plate/mount/slot/ROI IDs NA
# ─────────────────────────────────────────────

for col in [
    "mount_id_inferred",
    "mount_id_source",
    "plate_key",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "bruker_roi_id",
]:
    if col not in df_bad.columns:
        df_bad[col] = pd.NA

# ─────────────────────────────────────────────
# 7) Recombine good + bad and write out
# ─────────────────────────────────────────────

df_linked = pd.concat([df_good, df_bad], ignore_index=True)

print("\nL6 — final df_linked shape:", df_linked.shape)
print("L6 — final link_source breakdown:")
print(df_linked["link_source"].value_counts())

print("\nL6 — sample plate/slot/ROI hierarchy with bruker_roi_id (good rows):")
print(
    df_good[
        [
            "roi_dir",
            "plate_date",
            "experiment_folder",
            "mount_id_inferred",
            "plate_id_filled",
            "fish_id",
            "slot_id_filled",
            "roi_folder",
            "roi_index_within_slot",
            "bruker_roi_id",
            "Date imaged",
            "date_mount",
            "link_source",
        ]
    ].head(30)
)

out_link = WORKING / "output_from_linking_v5.csv"
df_linked.to_csv(out_link, index=False)
print("\nL6 — wrote linking_v5 output to:", out_link)
print("L6 — done.")

L6 — rows with missing plate_date AFTER all fallbacks: 11
L6 — wrote rows with missing plate_date to: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/linking_v5_missing_plate_date_rows.csv
L6 — will carry these rows through with NA mount/plate/slot IDs (manual review).
L6 — inferred plates with >6 fish_id (should be 0): 0

L6 — final df_linked shape: (1082, 46)
L6 — final link_source breakdown:
link_source
sheet         927
unmatched     112
date_match     43
Name: count, dtype: int64

L6 — sample plate/slot/ROI hierarchy with bruker_roi_id (good rows):
                                              roi_dir plate_date  \
0   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250428   
1   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250428   
2   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
3   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
4   /clusterfs/vast/abcabc/Korra_Foundation/202504...   20250429   
5   /clu